## **Sustainability and the Cost of Intelligence**

---

In [ ]:
import zipfile
import os
import shutil

zip_path = "NB03_assets.zip"
folder_name = os.path.splitext(zip_path)[0]

# Extract to a temporary location first
temp_extract = "temp_extract"

with zipfile.ZipFile(zip_path, 'r', allowZip64=True) as zip_ref:
    zip_ref.extractall(temp_extract)

# Check what's in the temp folder
contents = os.listdir(temp_extract)

# If there's only one folder and it matches our desired name, move its contents up
if len(contents) == 1 and os.path.isdir(os.path.join(temp_extract, contents[0])):
    # Move the inner folder to the desired location
    shutil.move(os.path.join(temp_extract, contents[0]), folder_name)
    # Remove the temp folder
    shutil.rmtree(temp_extract)
else:
    # Otherwise, just rename the temp folder
    shutil.move(temp_extract, folder_name)

print(f"Extracted to: {folder_name}")
print(f"Contents: {os.listdir(folder_name)[:5]}")  # Show first 5 items

### **0. Notebook Overview and Learning Goals**

In this notebook, we shift our attention from *what models can do* to *what it costs to make them work*. Modern deep learning relies on increasingly large models, massive datasets, and powerful computational infrastructures. While these advances enable impressive capabilities, they also carry significant environmental and social implications. Our goal in this notebook is to develop a clear, quantitative, and ethically grounded understanding of the **energy, compute, and carbon footprint** associated with contemporary AI systems.

We approach this topic by examining real data from recent large-scale models, exploring how electricity is consumed across the AI lifecycle, and considering techniques that help reduce these impacts. We use visual evidence, energy estimates, and simplified mathematical reasoning to build an intuition for why AI sustainability has become a central concern in research and policy.

**Learning Goals**

By the end of this notebook, we should be able to:

- **Explain why modern AI consumes so much compute** and how scaling laws drive ever-larger architectures.
- **Interpret key terms** such as parameters, FLOPs, GPU-hours, TDP (thermal design power), PUE (power usage effectiveness), and carbon intensity.
- **Compare the electricity and carbon footprint** of well-known models such as GPT-3, PaLM, LLaMA, and diffusion models.
- **Understand the lifecycle of AI energy use** — how training, fine-tuning, and inference contribute differently.
- **Analyze the role of data centers**, cooling systems, geographic locations, and clean energy availability.
- **Learn techniques for efficient AI**, including pruning, quantization, distillation, and sparse mixture-of-experts.
- **Reflect on the ethical tensions** between innovation, environmental sustainability, and global access to compute resources.

This notebook prepares us for broader discussions about **responsible AI infrastructure**, **green AI practices**, and the global consequences of scaling machine intelligence. It also sets the foundation for NB04, where we turn to the **future of AI**, exploring emerging architectures, societal implications, and governance of increasingly powerful systems.


### 🌍 **1. Why Compute Matters: From Scaling Laws to Environmental Cost**

Modern AI systems — especially large models such as GPT-3, PaLM, LLaMA, and DeepSeek — rely on an unprecedented amount of computational power. In this section, we explore *why* compute has grown so rapidly over the past decade, how scaling laws drive this growth, and what this means for the environment.

🧠 **1.1 The Rise of Compute in Modern AI**

Over the past 10–12 years, the amount of compute used to train frontier models has been growing **exponentially**, in some analyses doubling every **6–9 months** and in others increasing by **4.4× per year**.  
This explosive growth is not accidental — it emerges from deep statistical principles about how model performance improves with more data, more parameters, and more training.

To understand the sustainability implications, we must first understand *why* compute keeps rising.

---

In [1]:
from IPython.display import VimeoVideo

# Bigger video
VimeoVideo("1148866354", h="3298dbabb7", width=700, height=450)  

📈 **1.2 Scaling Laws: Why Bigger Models → Lower Error → Higher Compute**

Scaling laws describe a predictable relationship between:

- **Model size** (parameters)  
- **Training dataset size**  
- **Training compute (FLOPs)**  
- **Resulting error** (loss, perplexity, accuracy, etc.)

In many domains, as we increase model or dataset size, the generalization error follows a **power-law decline** — steady, predictable, and continuous. However, this improvement comes with sharply increasing computational cost.

<img src="NB03_assets/10_Common shape of a scaling law, taken from_epoch_ai_blog_scaling_laws_literature_review.png" width="400">

This figure [1] shows three key regions:

1. **Small-Data Region** — Adding more data does little.
2. **Power-Law Region** — Error decreases smoothly as data increases.
3. **Irreducible Error Region** — No amount of data or compute helps.

Most modern AI breakthroughs occur in the **power-law region**, where performance improves *only* if we push compute upward by orders of magnitude.

In [2]:
from IPython.display import VimeoVideo

# Bigger video
VimeoVideo("1148866382", h="3298dbabb7", width=700, height=450) 


⚙️ **1.3 Compute Has Been Growing Faster Than Moore’s Law**

The leap from AlexNet (2012) to PaLM/DeepSeek/GPT-4 required not just more powerful hardware, but massive *scale* in cluster size, dataset size, and training FLOPs.

**Training Compute of Notable Models (Epoch AI)** [2]

<img src="NB03_assets/16_FLOPs_inNotableAI_Models_post2010_from_epoch_Ai.png" width="800">

This dataset reveals:

- Compute is increasing at **4.4× per year** — dramatically faster than the traditional gains predicted by Moore’s Law.
- Each new frontier model tends to use **10× to 100×** more compute than the generation before it.

This trend has direct implications for sustainability because compute is tightly coupled to **electricity consumption**, **cooling**, and ultimately **carbon emissions**.

🔍 **1.4 Training Compute in Deep Learning More Broadly**

Not only LLMs, but **all deep learning subfields** — vision, speech, robotics, reinforcement learning — show similar exponential rises.

**Training Compute Across Domains** [2]

<img src="NB03_assets/22_training_compute_DL_Models_from_epoch_AI.png" width="800">

This chart reinforces that:

- Across all AI domains, training FLOPs have skyrocketed since 2010.
- Some state-of-the-art models today require **10²⁴–10²⁶ FLOPs**, levels unimaginable just a decade ago.  

Such increases directly translate into *massive energy requirements*.

🧮 **1.5 Parameters: The Other Driver of Compute**

More parameters generally → more compute → more energy.  
The number of parameters in frontier models has risen from millions (AlexNet) to billions (ResNet variants, transformers) to **trillions** (GPT-4-scale architectures, Mixture-of-Experts models).

**Number of Parameters Over Time (Our World in Data)** [3]

<img src="NB03_assets/15_Parameters in notable artificial intelligence systems_from_ourworldindata_org.png" width="800">

This visual makes it clear:

- Around 2012–2015: models reached the **10M–100M** scale.  
- Around 2018–2020: models crossed the **1B–10B** threshold.  
- Today: **100B–1T+** models are normal, with MoE models scaling to trillions of active/conditional parameters.

This growth explains why training demands enormous compute clusters and entire data-center infrastructures.

🔗 **1.6 Linking Parameters to Training Compute**

Model size (parameters) and dataset size are major contributors to compute.  
The relationship can be visualized through *training compute vs. parameter count* [3]:

<img src="NB03_assets/14_training comparisons and parameters_from_ourworldindata_org.png" width="800">

This chart shows:

- Compute rises **superlinearly** with parameter count  
- Many models cluster along predictable scaling trajectories  
- Larger models are disproportionately more expensive to train

🌡️ **1.7 Why This Matters for Sustainability**

The exponential rise in compute does not occur in a vacuum.  
It drives:

- Enormous electricity demand  
- Growth in data center capacity  
- Increased cooling requirements  
- Greater carbon emissions depending on grid sources  
- Hardware manufacturing and lifecycle impacts

Before we can discuss solutions such as pruning, quantization, or efficient hardware, we must first grasp the **computational forces** that push AI toward greater environmental cost.

The key message:

> **The compute explosion makes sustainability a central concern — not a secondary detail.**

---

**✅ ALERT: You are now ready to answer the Multiple Choice Questions for this section!**

---

In [3]:
from IPython.display import VimeoVideo

# Bigger video
VimeoVideo("1148866416", h="3298dbabb7", width=700, height=450)  

### ⚡ **2. How Big Models Consume Power: Parameters, FLOPs, and Energy**

To understand the environmental footprint of modern AI systems, we need to understand *how* large models consume electricity. This requires breaking down several fundamental concepts—**parameters**, **FLOPs**, **compute throughput**, **hardware efficiency**, **data center overhead**, and **carbon intensity**.

This section builds a clear vocabulary so we can quantify and interpret the energy demands of frontier models like GPT-3, GPT-4, GLaM, and PaLM.

🔑 **2.1 Key Concepts: Parameters, FLOPs, TDP, PUE, Carbon Intensity**

The table below summarizes essential indicators used in sustainability and compute analysis.

**Definitions**

| Term | Meaning |
|------|---------|
| **Parameter** | A learned weight in a neural network. Big models = billions/trillions of parameters. |
| **FLOP** | A single *Floating Point Operation* (e.g., addition/multiplication). Training requires **10²³–10²⁶ FLOPs** for frontier models. |
| **FLOPs/s (Throughput)** | How many floating-point operations the hardware can perform per second. Determines training time. |
| **TDP (Thermal Design Power)** | Approximate sustained power draw of a GPU/TPU during full load (e.g., A100 ≈ 400W). |
| **PUE (Power Usage Effectiveness)** | Data center overhead metric. PUE = 1.0 means perfect efficiency; 1.2–1.6 is typical. |
| **Carbon Intensity** | How “clean” the electricity source is (e.g., France ≈ 0.05 kg CO₂/kWh, U.S. ≈ 0.4–0.6 kg CO₂/kWh). |

These terms will help us move from *compute* → *power* → *energy* → *carbon impact*.

🧮 **2.2 From Compute to Energy: A Simple Relationship**

Training energy can be approximated using:

```text
    Energy (kWh) = Power Draw (kW) × Training Time (hours) × PUE
```
Or, in terms of total compute:

```text
    Energy ≈ (Total FLOPs / Hardware FLOPs-per-second) × Power × PUE
```
This expresses several important relationships:
- More FLOPs → longer training → more energy
- Less efficient hardware → more power draw
- Higher PUE → higher electricity overhead
- Location matters (clean vs. fossil electricity source)

This helps students appreciate that the environmental impact is not just about model size — but also *duration*, *infrastructure*, and *electricity source*.

🔋 **2.3 Power Draw Is Increasing Every Year**

Frontier models require not only more **compute**, but also more **power delivery**, **cooling**, and **data center capacity**.

**Training Compute & Power Requirements (Epoch AI)** [2]

<img src="NB03_assets/17_Power_requirement_doubling_annualy_from_epoch_Ai.png" width="700">

Key takeaways:
- Power requirements for training frontier models are increasing at **~2.1× per year**.
- GPT-3 levels of compute now appear relatively modest compared to GPT-4, Gemini Ultra, Grok-4, Llama-3 400B, etc.
- The total power required for a full training run can reach **megawatts** for weeks at a time.

This doubling pace is one of the strongest arguments for focusing on **sustainable infrastructure**.

🔥 **2.4 How FLOPs → Power: The Compute–Energy Relationship**

We next connect the earlier compute trends to the energy discussion.

**Training Compute of Notable Models (Revisited)** [2]

<img src="NB03_assets/16_FLOPs_inNotableAI_Models_post2010_from_epoch_Ai.png" width="700">

Why this matters here:
- The exponential rise in training compute (4.4× per year) directly inflates **total energy** consumption.
- Compute is performed on thousands of GPUs/TPUs simultaneously, raising:
-- instantaneous power draw (MW scale)
-- heat generation
-- cooling overhead
-- data center PUE load

We emphasize that training compute is not abstract — it maps concretely to **electricity usage**, **energy bills**, and **carbon emissions**, depending on energy mix.

⚙️ **2.5 Putting It Together: Training Energy for Frontier Models**

If we know:
- **Compute needed** (FLOPs)
- **Hardware throughput** (FLOPs/s per GPU/TPU)
- **Cluster size** (number of accelerators)
- **Average power draw** (TDP × utilization)
- **Data center PUE**

Then we can estimate training energy and emissions.

Example: 
```text
        Compute required:        3×10²³ FLOPs  
        Hardware:                2000 GPUs at 400W each  
        Power draw:              ≈ 0.8 MW  
        Training duration:       30 days  
        PUE:                     1.3  
        Energy = 0.8 MW × 720 h × 1.3  
            ≈ 748 MWh
```


---

**✅ ALERT: You are now ready to answer the Multiple Choice Questions for this section!**

---

### ⚡ **3. Case Study: Electricity Use of GPT-3, PaLM, LLaMA, BLOOM, DeepSeek**

In this section, we move from *theory* to *real numbers*. Students often understand sustainability challenges best when they see concrete electricity consumption values for real models — including how these values differ by architecture, by region, and by energy mix.

This case study organizes the evidence into three perspectives:

1. **Total electricity consumption**  
2. **Architectural efficiency differences (dense vs MoE)**  
3. **CO₂ emissions variation due to carbon intensity of national grids**

These examples will prepare us for the sustainability discussions in later sections.

🔌 **3.1 Electricity Consumption of Leading Large Models**

The table below [4] summarizes estimated electricity use for several frontier models:

<img src="NB03_assets/03_table_Params_and_electricity_consumption_evaluations_for_leading LLMs_from_Article_systematic_review_of_electricity_demand_forLLMs.png" width="500">


**Key Observations (Narrative Explanation)**

- **BERT** consumes only ~1.5 MWh in training — small compared to LLMs.
- **GPT-2** is also small-scale (~1.7 MWh).
- **GPT-3** jumps dramatically to **~1,287 MWh**, showing nonlinear scaling in both FLOPs and energy.
- **GPT-4** appears in multiple estimates (≈ 51,000–62,000 MWh), reflecting:
  - larger model sizes  
  - longer training cycles  
  - extreme cluster sizes  
- **PaLM** at 540B parameters consumed **~3,436 MWh** — more than GPT-3 despite architectural differences.
- **LLaMA-2 (70B)** consumes much less due to efficiency improvements and reduced training scope.

This table often shocks a lot of people: training a single frontier model can consume **as much electricity as hundreds of households use in a year**.

🧠 **3.2 Dense vs MoE Architectures: Why Mixtral & GLaM Are More Efficient**

Mixture-of-Experts (MoE) models activate only a fraction of the parameters per token, which drastically reduces training FLOPs and electricity demand.

We can visualize this with [4]:

![](NB03_assets/09_Comparisons among GPT-3, GLaM, and Mixtral 8 × 7B_from_Article_systematic_review_of_electricity_demand_forLLMs.png)

- **GPT-3 (dense)**: 175B parameters, very high electricity consumption.
- **GLaM (Mixture-of-Experts)**:  
  - 1.162 trillion total parameters on paper  
  - Only ~97B active per forward pass  
  - ≈ **3× lower energy** than GPT-3
- **Mixtral 8×7B (Sparse MoE)**:  
  - Activates only 2 of 8 experts → **14B active params**  
  - Electrical cost is *far lower* than a dense 50B+ model.

➡️ *Architectural choice is one of the strongest levers for reducing energy demand.*


In [4]:
from IPython.display import VimeoVideo

# Bigger video
VimeoVideo("1148866428", h="3298dbabb7", width=700, height=450) 

🌍 **3.3 Location Matters: CO₂ Emissions Depend on Grid Carbon Intensity**

Even if two models consume the same **electricity (MWh)**, their **carbon emissions (ton CO₂)** can differ dramatically depending on the national grid where training occurs.

We show this with [4]:

<img src="NB03_assets/04_Impact of carbon intensity on emissions for three representative LLMs_from_Article_systematic_review_of_electricity_demand_forLLMs.png" width="500">

- GPT-3 trained in the U.S. grid → **~552 tons CO₂**  
- DeepSeek-V3 trained on China’s grid → **~550 tons CO₂**, despite different MWh values  
- BLOOM-176B trained in France → **only ~24.7 tons CO₂** due to clean nuclear-led electricity mix

**Insight**

- **Electricity consumption alone ≠ environmental impact.**  
- What matters is:  
    ```text
      CO₂ Emissions = Electricity (kWh) × Carbon Intensity (kg CO₂/kWh)
    ```
- Low-carbon grids (France, Norway, Sweden) drastically reduce AI training emissions.

This is a crucial point for policy and responsible AI infrastructure.

📝 **3.4 Why This Case Study Matters**

By comparing GPT-3, PaLM, LLaMA-2, GLaM, and DeepSeek-V3, students learn that:
- Model architecture (dense vs MoE) can reduce electricity by 3×–10×.
- Geographic location can reduce CO₂ emissions by 10×–20× for the same compute.
- Different models consume **1 MWh → 60,000 MWh**, illustrating the vast scale gap.

These numbers transform sustainability from an abstract concern into a quantifiable engineering challenge.

---

**✅ ALERT: You are now ready to answer the Multiple Choice Questions for this section!**

---

### ⚡ **4. Fine-Tuning vs Training vs Inference: Why Inference Dominates Global Electricity**

In earlier sections we saw that **training frontier models** (GPT-3, PaLM, LLaMA-70B, DeepSeek-V3) can consume massive electricity.  
But one crucial insight often surprises students:

> **Inference — not training — accounts for 60–90% of total lifetime electricity consumption of large AI systems.**

This section explains *why*, using lifecycle diagrams and data tables drawn from the systematic review of LLM electricity demand.

In [5]:
from IPython.display import VimeoVideo

# Bigger video
VimeoVideo("1148866456", h="3298dbabb7", width=700, height=450) 

🔄 **4.1 The LLM Lifecycle: Training → Fine-Tuning → Inference**

A typical model has three energy-consuming stages:

1. **Pretraining**  
   - Most compute-intensive step  
   - Happens only *once*  
   - Requires massive datasets  
   - Uses enormous GPU/TPU clusters  
   - Electricity use: **very high but infrequent**

2. **Fine-tuning / Alignment**  
   - Smaller datasets  
   - Fewer epochs  
   - Often done multiple times for different product versions  
   - Electricity use: **moderate and episodic**

3. **Inference (deployment)**  
   - Serving millions–billions of queries  
   - Happens *continuously*  
   - Even if each inference is small, global demand is huge  
   - Electricity use: **dominates the model’s lifetime footprint**

We visualize this using the lifecycle diagram [4]:

<img src="NB03_assets/01_charac_between_training_and_inference_in_one_LLM_lifecycle_from_Article_systematic_review_of_electricity_demand_forLLMs.png" width="700">

- Training uses enormous compute *per hour*, but happens only occasionally.  
- Inference uses very little compute *per request*, but happens **millions of times per hour**, every hour, globally.  
- When aggregated, inference can consume **10× more electricity than training** over the model’s lifetime.

This is the same pattern websites follow: development is expensive, but hosting traffic is what drives long-term cost.

📊 **4.2 Comparing Energy Use Across Training, Fine-Tuning, and Inference**

The following table [4] provides a structured comparison of the three phases:

![](NB03_assets/05_table_Comparisons_of_energy_demand_by_fine-tuning_vs_training_and_inference_from_Article_systematic_review_of_electricity_demand_forLLMs.png)

**Key takeaways**

**Training**
- Highest compute per hour  
- Longest continuous usage of GPUs  
- Electricity demand: **hundreds to thousands of MWh**  
- Occurs rarely (once per major model version)

**Fine-Tuning**
- Uses only **1–5% of training compute**  
- Electricity demand typically: **10–100 MWh**  
- Run multiple times for specific tasks (instruction tuning, safety alignment)

**Inference**
- Smallest per-operation energy  
- But *orders of magnitude more frequent*  
- Electricity demand depends on:
  - number of users  
  - token generation length  
  - batch sizes  
- Over months/years, inference becomes the **dominant electricity sink**

**The formula behind this pattern**

```text
      Total Electricity = Σ (energy per operation × number of operations)
```

Even if inference energy per token is tiny (~0.1–2 mJ/token), a model serving billions of tokens per day accumulates huge electricity use.

🌎 **4.3 Why Inference Dominates Global AI Electricity Use**

From a sustainability standpoint, this is critical:

**Reason 1 — Training happens a handful of times**

A model like GPT-4 may be trained only:

- once during development
- plus a handful of fine-tuning cycles

Thus the training energy, while large, is bounded.

**Reason 2 — Inference is continuous**

Every chatbot query, image generation, translation, summarization, or recommendation triggers GPU computation.

A single model used by:

- 50 million users daily
- generating billions of tokens per day

… produces electricity usage that **dwarfs** its training stage.

**Reason 3 — Deployment scale amplifies consumption**

Inference clusters often run:
- 24/7
- across thousands of GPUs
- across multiple regions (US, EU, Asia)

This is fundamentally different from academic training cycles.

**Reason 4 — Many models share one training run, but inference is per-user**

Training produces *one model artifact*.
Inference responds to *infinite user inputs*.
Therefore, inference grows linearly with usage.

❗ **4.4 Pedagogical Summary**

Students must understand this before sustainability decisions can be evaluated:

| **Stage**	| **Electricity per Hour** |	**Frequency** |	**Long-Term Impact** |
|-----------|--------------------------|----------------|------------------------| 
| **Training** |	Extremely high |	Rare |	Moderate |
| **Fine-tuning** |	Medium |	Occasional |	Small |
| **Inference** |	Low per request |	Constant, large scale |	Dominant |

**Inference, not training, is the primary sustainability bottleneck for AI deployment.**

This insight shifts ethical discussions from “Don’t train big models” to:

> **“How do we deploy models efficiently and responsibly?”**

---

**✅ ALERT: You are now ready to answer the Multiple Choice Questions for this section!**

---

### 🏭 **5. Data Centers, Cooling, and Infrastructure: Why Efficiency Matters**

When discussing the sustainability of AI, we often focus on **models**, **parameters**, and **compute**.  
But the true environmental impact of AI cannot be understood without zooming out to the **infrastructure** that powers machine learning systems worldwide.

Modern AI runs inside massive data centers — industrial-scale complexes filled with GPUs, TPUs, networking equipment, and enormous cooling systems.  
Electricity is consumed not only by computation but by *every layer* of the ecosystem.

This section introduces students to the broader picture:  
**hardware → data center → electric grid → society**.

🌐 **5.1 The Four-Tier Ecosystem of Electricity Demand**

AI energy demand emerges from interconnected layers.  
The following diagram illustrates a **bottom-up framework** of the entire system [4]:

![]("NB03_assets/06_Bottom_up_framework_with_four_tiers_of_challenges_and_their_factors_influencing_electricity_demand of LLMs_from_Article_systematic_review_of_electricity_demand_forLLMs.png)


This framework shows that electricity use is not determined solely by model size. It is shaped by:

1. **Model-level factors**  
   - Parameter count  
   - Training duration  
   - Inference load  
   - Hardware efficiency (FLOPs/Watt)

2. **Data center-level factors**  
   - Power distribution  
   - Cooling efficiency  
   - Network architecture

3. **Grid-level factors**  
   - Carbon intensity of electricity  
   - Renewable vs fossil fuel mix  
   - Local energy policy

4. **Societal-level factors**  
   - User demand  
   - Global AI adoption  
   - Market and regulatory pressures

This layered viewpoint helps students understand that *AI is an environmental system, not just a model.*

⚡ **5.2 Carbon Intensity: Why Training in One Country Can Emit 3–10× More CO₂**

Not all electricity is equal.  
The same GPU-hour can emit dramatically different CO₂ depending on the **regional power mix**. [4]

<img src="NB03_assets/07_Power capacity and carbon intensity of top ten data center markets_from_Article_systematic_review_of_electricity_demand_forLLMs.png" width="600">


- Countries with **renewable-heavy grids** (e.g., Norway, France) emit far less CO₂ per kWh.  
- Regions relying on **coal or gas** (e.g., parts of the US, China, India) generate much higher emissions.  
- The location of a training run can change total carbon footprint by **up to an order of magnitude**.

This means:
- A 1,000 MWh training run in Norway produces *far less* CO₂ than the same run in coal regions.  
- Tech companies increasingly choose data center locations to reduce emissions.

📈 **5.3 Data Centers Demanding Higher Electricity Share Globally**

AI-driven data centers are rapidly increasing their share of global electricity consumption [5].

<img src="NB03_assets/12_DataCentersDemandingHigherPowerShare_from_andthewest_stanford_edu.png" width="700">


- Data center electricity demand is projected to rise sharply → AI is the primary driver.  
- This growth risks straining national grids in regions without strong infrastructure.  
- Local communities may face increased electricity prices or shortages due to hyperscale AI deployment.

This highlights the ethical responsibility of organizations building large models.

❄️ **5.4 Why Cooling Is a Major Part of AI’s Energy Footprint**

A surprising fact for many students:

> **Up to 30–50% of data center electricity goes to cooling, not computation.**

Training models heats GPUs significantly, so data centers invest heavily in cooling solutions.

**Comparing Cooling Technologies** [6]

Air Cooling vs Liquid Cooling

<img src="NB03_assets/11_Liquid_Cooling_vs_Air_Cooling_The Data Center Debate in 2025_microscan_co_in.png" width="500">

**Air cooling**
- Cheaper, simpler  
- Less energy efficient  
- Struggles with modern GPU thermal loads

**Liquid cooling**
- More energy efficient  
- Better for dense GPU clusters  
- Essential for frontier AI labs

The shift from **PUE ≈ 1.6 → PUE ≈ 1.1** (Power Usage Effectiveness) represents *massive* electricity savings.

**Immersion Cooling**

A more advanced technique gaining traction in AI labs like Google, Microsoft, Baidu [7]:

<img src="NB03_assets/13_AI_data_centers_from_thundersaidenergy_com.png" width="700">

**Immersion cooling**:
- Submerges hardware in thermally conductive fluid  
- Greatly reduces cooling costs  
- Extends component lifespan  
- Enables higher training throughput with lower electricity per FLOP

🌱 **5.5 Why Infrastructure Efficiency Matters for Responsible AI**

Even if a model is compute-heavy, responsible infrastructure can dramatically reduce emissions:

- Efficient cooling reduces electricity waste  
- Renewable-powered data centers reduce carbon footprint  
- Low-PUE designs amplify sustainability gains  
- Locating compute in clean-energy regions multiplies impact

In short:

> **Model efficiency matters — but infrastructure efficiency matters equally.**

By the end of this section, students should understand:

- Electricity is consumed by GPUs *and* the systems around them.  
- Cooling is a major contributor to energy footprint.  
- Training location heavily affects carbon emissions.  
- Infrastructure design can make AI dramatically more sustainable.

This knowledge prepares students for Section 6, where we explore *model efficiency methods* such as pruning, quantization, and distillation.

---

**✅ ALERT: You are now ready to answer the Multiple Choice Questions for this section!**

---

In [6]:
from IPython.display import VimeoVideo

# Bigger video
VimeoVideo("1148866479", h="3298dbabb7", width=700, height=450) 

### 🧩 **6. Making AI More Efficient: Compression, Pruning, Quantization, MoE**

In earlier sections, we explored why modern AI models consume large amounts of energy:  
billions of parameters → trillions of FLOPs → massive data center loads.  
In this section, we shift from *diagnosis* to *solutions*.  

We explore practical techniques that researchers and engineers use to **shrink**, **optimize**, and **accelerate** deep learning models, thereby reducing electricity use and environmental impact.

Each technique corresponds to a measurable reduction in:
- **parameter count**
- **model size (GB)**
- **compute required per inference**
- **energy consumed per training step**

🌱 **6.1 Why Model Compression Matters**

Deep learning models vary dramatically in size [8].

<img src="NB03_assets/18_Number-of-parameters-for-commonly-used-neural-network-architectures_fromResearchgate_RestrictedBoltzmannMachine.png" width="600">

This chart shows that early CNN models such as **VGG-16** contain *many more* parameters than more modern, optimized architectures such as **ResNet** or **MobileNet**.

**Key Takeaways**
- Large parameter counts → high memory footprint → more energy for every forward/backward pass.
- Modern architectures achieve **similar or better accuracy** with **10× fewer parameters**.
- Efficient design is a critical sustainability strategy.

✂️ **6.2 Pruning: Removing Unnecessary Neurons and Weights**

Pruning is one of the simplest—and most powerful—ways to shrink models.

The idea:  
**many parameters contribute almost nothing to the final output**, especially in overparameterized networks [9].

<img src="NB03_assets/19_before_after_prunning_from_medium_modelcompressing_prunning.png" width="500">

**How Pruning Works**
1. Train a large model normally.
2. Identify weights or neurons with very small magnitude (almost unused).
3. Remove them (“prune”).
4. Optionally fine-tune the remaining network.

**Benefits**
- Smaller model size  
- Faster inference  
- Lower FLOPs → reduced electricity use  
- Can often keep accuracy within 1–2%

**Intuition for Understanding**
A pruned network behaves like a “compressed brain,” keeping only the strongest connections.

⚙️ **6.3 Quantization: From 32-bit to 8-bit (or even 4-bit!) Computation**

Modern models typically use **32-bit floating-point numbers (FP32)**.  
Quantization shrinks these to **8-bit integers (INT8)** or lower.

<img src="NB03_assets/20_Quantization_from_scaler_com_topics_quantization-and-pruning.png" width="500">

**What Quantization Does** [10]
- Reduces model size by **4×**  
- Reduces memory bandwidth by **4×**  
- Speeds up operations dramatically on optimized hardware (e.g., NVIDIA Tensor Cores)

**Example**: A 1 GB FP32 model → 250 MB INT8 model.

**Why It Matters**
Energy use in AI is proportional to both:
- the number of operations  
- the *precision* of those operations

Lower precision means **less electricity per computation**.

🧠 **6.4 Knowledge Distillation: Teaching a Smaller Student Model**

Instead of compressing a model directly, we can train a **small model** to mimic a **large model**.

**How Distillation Works**
1. Train a large, powerful **teacher model**.
2. Use its soft predictions to train a smaller **student model**.
3. Student learns “teacher-like behavior” with far fewer parameters.

**Benefits**
- Student can be **10× smaller**  
- Student often achieves **90–95%** of teacher accuracy  
- Great for deployment on phones and edge devices  

This is used widely:
- DistilBERT  
- TinyBERT  
- DistilGPT  
- MiniLM  

🪢 **6.5 Low-Rank Adaptation (LoRA) & Parameter-Efficient Fine-Tuning (PEFT)**

Frontier LLMs often have **tens or hundreds of billions of parameters**.  
Fine-tuning all weights is computationally expensive.

PEFT methods update only a *tiny fraction* of parameters.

**LoRA Intuition**
Instead of updating the full weight matrix  
$ W \in \mathbb{R}^{d \times k} $,  
LoRA learns **two much smaller matrices** whose product approximates the update.

This reduces training cost by **10–100×**.

**Why This Matters for Sustainability**
- Less GPU memory needed  
- Less backpropagation compute  
- Faster training  
- Works extremely well: used in many LLM fine-tuning pipelines

🔀 **6.6 Mixture-of-Experts (MoE): Sparse Compute Instead of Dense Compute**

MoE architectures (e.g., **GLaM**, **Mixtral**) do something clever:

> **Instead of activating *all* weights all the time, they activate only a small “expert subset.”** [4]

<img src="NB03_assets/09_Comparisons among GPT-3, GLaM, and Mixtral 8 × 7B_from_Article_systematic_review_of_electricity_demand_forLLMs.png" width="500">

**Why MoE Saves Energy**
- Sparse activations → fewer FLOPs per token  
- Models behave like “multiple specialists” instead of one giant generalist  
- Parameter count stays high (e.g., 1T+), but *effective compute* stays manageable

**Example**
- GPT-3: dense, 175B parameters  
- GLaM: MoE, 1.2T parameters but **only uses ~97B parameters per forward pass**

This can reduce training electricity by **3–5×**, depending on routing efficiency.

🌍 **Key Takeaways**

By the end of this section, students should understand:

- Why shrinking models reduces electricity consumption.  
- How pruning, quantization, distillation, and PEFT make real models more efficient.  
- Why MoE architectures represent a fundamental shift in sustainability.  
- That responsible AI includes *engineering models that waste less energy*.

---

**✅ ALERT: You are now ready to answer the Multiple Choice Questions for this section!**

---

### 🌍 **7. Reflection: Ethical, Social, and Environmental Responsibility**

In this final reflective section, we step back from the technical content and consider the *ethical*, *social*, and *environmental* questions that surround large-scale AI.  
As models become larger and more energy-intensive, we must ask whether all forms of scaling are justified — and who bears the cost.

We are no longer thinking about FLOPs and GPU-hours alone, but about the **societies, ecosystems, and power structures** that these models affect.

**7.1 Should We Always Scale?**

Over the past decade we have seen an accelerating trend:  
*more parameters → more data → more energy → better performance.*

Scaling laws suggest that performance continues to improve—sometimes dramatically—as we increase compute.  
But scaling also brings **environmental costs**, **financial costs**, and **accessibility inequalities**.

> **Key reflection:**  
> Just because a model *can* be made larger, does that mean it *should*?

Some tasks genuinely require massive models; others may benefit from careful engineering of smaller architectures. Students should reflect on where to draw this line.

**7.2 What Is a “Responsible Parameter Count”?**

This is not only a technical question — it is a governance question.

A model with **100 million parameters** may solve a problem effectively while remaining energy-efficient.  
A model with **100 billion parameters** may perform better, but:
- require enormous electricity during training,
- raise high financial barriers to entry,
- centralize AI development among a handful of organizations.

We must ask:

> **Where is the ethical trade-off between performance and sustainability?**  
> **When does scaling become irresponsible?**

Encourage students to imagine they are designing an AI system under real-world constraints:  
What parameter count would *they* choose?

**7.3 Global Electricity Equity and the AI Divide**

Electricity is not distributed equally across the world.  
Training frontier models often requires:
- abundant, cheap, and stable electricity,
- high-end data centers,
- access to GPUs/TPUs prioritized for wealthy nations and corporations.

Meanwhile, many regions — especially in the Global South — struggle with:
- unstable electricity grids,
- energy deficits,
- limited compute access.

This raises profound questions:

- Is it ethical for high-income nations to use vast amounts of energy for AI while others face shortages?
- Does large-scale AI widen global inequality by restricting participation to wealthy institutions?

These questions connect directly to sustainability, justice, and long-term governance.

**7.4 Points to ponder**

Use these as discussion or writing prompts in the notebook:

1. If scaling improves model accuracy, at what point should we stop?
2. Should AI companies be required to publish the electricity and carbon footprint of model training?
3. Is it ethical to deploy a 100B-parameter model for a task that a 200M-parameter model could solve adequately?
4. How should global electricity inequality influence the design of AI systems?
5. What would a “responsible compute budget” for AI research look like?

These questions help students think beyond technical efficiency toward **ethical responsibility**.

---

### 🔚 **8. Summary & Transition to NB04**

We end NB03 by summarizing the journey:

**What we learned in this notebook**

- Modern AI models consume enormous compute, which translates to electricity and carbon emissions.  
- FLOPs, GPU-hours, carbon intensity, and cooling systems all contribute to the environmental footprint.  
- Training is expensive, but inference often dominates lifetime electricity use.  
- Model compression methods — pruning, quantization, distillation, LoRA, MoE — offer pathways to **Green AI**.  
- Ethical questions arise around scaling, global energy equity, and responsible compute.

This notebook connects technical knowledge with environmental and social awareness, preparing students to think critically about *sustainable AI development.*

**What comes next in NB04**

In the final notebook of Project 12, we shift from conceptual understanding to **hands-on evaluation and profiling**.

Students will learn to:
- measure the inference **latency**, **memory use**, and **energy estimate** of models,
- compare lightweight vs. heavyweight architectures,
- reason about deployment trade-offs (mobile vs cloud),
- connect sustainability to **real implementation choices**.

NB04 will help students see that responsible AI is not just theory —  
it is **something we measure, design for, and intentionally optimize**.

---

**Bibliography**

[1] Pablo Villalobos (2023), "Scaling laws literature review". Published online at epoch.ai. Retrieved from: 'https://epoch.ai/blog/scaling-laws-literature-review' [online resource; Last accessed on 10 Dec 2025] </br>
[2] epoch.ai </br>
[3] https://ourworldindata.org/grapher/ai-training-computation-vs-parameters-by-domain [Last accessed on 10th Dec 2025] </br>
[4] Ji, Z., & Jiang, M. (2026). A systematic review of electricity demand for large language models: evaluations, challenges, and solutions. *Renewable and Sustainable Energy Reviews*, 225, 116159.</br>
[5] https://andthewest.stanford.edu/2025/thirsty-for-power-and-water-ai-crunching-data-centers-sprout-across-the-west/ [Last Accessed on 10th Dec 2025]</br>
[6] https://www.microscan.co.in/post/liquid-cooling-vs-air-cooling-the-data-center-debate-in-2025 [Last Accessed on 10th Dec 2025]</br>
[7] https://thundersaidenergy.com/2024/04/25/cool-customers-ai-and-data-center-cooling/ [Last Accessed on 10th Dec 2025]</br>
[8] hSobczak, S., Kapela, R., McGuinness, K., Swietlicka, A., Pazderski, D., & O’Connor, N. E. (2021). Restricted Boltzmann machine as an aggregation technique for binary descriptors. *The Visual Computer*, 37(3), 423-432.
[9] https://medium.com/@shmilysyg/model-compression-pruning-quantization-distillation-and-binarization-7710ac954567 [Last accessed on 10th Dec 2025]</br>
[10] https://www.scaler.com/topics/quantization-and-pruning [Last accessed on 10th Dec 2025]</br>